In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

# for working with .toml
import tomli_w

%load_ext autoreload
%reload_ext autoreload
%autoreload 2
# this is important to be able to re-import the module after making modifications to the zavolab_pyutils code on Scicore

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("ONT_seq_PAIP1_Hondele.scicore.env",override=True)

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "drosophila_annotation_dir": os.getenv("DROSOPHILA_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "configs_dir": os.getenv("CONFIGS_DIR"),
    "pod5_dir": os.getenv("POD5_DIR"), # nanopore-specific
    "dorado_models_dir": os.getenv("DORADO_MODELS_DIR"), # nanopore-specific
    "nanoflowz_dir": os.getenv("NANOFLOWZ_DIR"), # nanopore-specific
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "drosophila_genome_file": os.getenv("DROSOPHILA_GENOME_FILE"),
    "drosophila_annotation_file": os.getenv("DROSOPHILA_ANNOTATION_FILE"),
    "dorado_executor": os.getenv("DORADO_EXECUTOR"), # nanopore-specific
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Prepare start samples

In [ ]:
paths_to_look = [
subdirs['pod5_dir']+'p41170_o41323_1/pod5_pass/',
]

pod5_file_paths = []
for path_ in paths_to_look:
    command = """find """+path_+""" -name '*.pod5' > """+subdirs['temp_dir']+"""pod5_files.tsv"""
    out = subprocess.check_output(command, shell=True)
    tmp = pd.read_csv(subdirs['temp_dir']+'pod5_files.tsv',delimiter="\t",
                                   index_col=None,header=None)
    pod5_file_paths.append(tmp)
pod5_file_paths = pd.concat(pod5_file_paths).reset_index(drop=True)
pod5_file_paths.columns = ['pod5']
pod5_file_paths['sample_id'] = pod5_file_paths.apply(lambda x:x['pod5'].split('/')[-2],1)

# we are only interested in barcodes 01-12
sel_sample_ids = range(1,13)
sel_sample_ids = [('barcode0'+str(elem) if len(str(elem))==1 else 'barcode'+str(elem)) for elem in sel_sample_ids]
pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['sample_id'].isin(sel_sample_ids)].reset_index(drop=True)

We will run nanoflowz independently for Hela (human cell line) and Drosophila samples

In [ ]:
# load sample metadata
barcodes_metadata_df = pd.read_csv(subdirs['metadata_dir']+'barcodes_metadata.tsv',delimiter="\t",
                                   index_col=None,header=0)
barcodes_metadata_df['sample_id'] = barcodes_metadata_df.apply(lambda x:'barcode'+('0' if x['Barcode number']<10 else '')+str(x['Barcode number']),1)
barcodes_metadata_df['organism'] = barcodes_metadata_df.apply(lambda x:'human' if ('human' in x['species'].lower()) else 'drosophila',1)

In [ ]:
pod5_file_paths = pd.merge(pod5_file_paths,barcodes_metadata_df[['sample_id','organism']],how='left',on='sample_id')

In [ ]:
# specify outdir and input samples .tsv for nanoflowz runs

organisms = ['human','drosophila']

for organism in organisms:
    WF_version = 'v1p4_ONT_PAIP1_'+organism+'_v1' # v1p4 at the beginning indicated Dorado version, while v1 at the end indicates the particular version of the analysis
    
    dir_path = subdirs['wf_runs_dir']+WF_version+'/'
    (Path(dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory for this version

    pod5_file_paths_cur = pod5_file_paths.loc[pod5_file_paths['organism']==organism].reset_index(drop=True)

    pod5_file_paths_cur[['sample_id','pod5']].to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

# Configuring nanoflowz execution

We've created a conda env "nextflow" to execute `nanoflowz`:

```bash
conda create --name nextflow bioconda::nextflow
conda activate nextflow
```

In [ ]:
# We installed `dorado` from ONT's github and got the path to executor:
print(file_paths['dorado_executor'])

In [ ]:
# We installed `nanoflowz` from github to this directory:
print(subdirs['nanoflowz_dir'])

In [ ]:
organisms = ['human','drosophila']

for organism in organisms:
    gpu_to_use = ('a100' if organism=='human' else 'rtx4090')
    
    WF_version = 'v1p4_ONT_PAIP1_'+organism+'_v1'
    dir_path = subdirs['wf_runs_dir']+WF_version+'/'
    
    run_dir = subdirs['wf_runs_dir']+WF_version+'/'
    
    # create a .toml file for polyA tail profiling
    polyA_toml_output_path = os.path.join(dir_path,'polyA_config.toml')
    (Path(polyA_toml_output_path).parent).mkdir(parents=True, exist_ok=True) # create subdir, just in case
    
    config_data = {
        "tail": {
            "tail_interrupt_length": 1
        }
    }
    with open(polyA_toml_output_path, "wb") as f:
        tomli_w.dump(config_data, f)
    
    # define pipeline configuration
    json_output_path = dir_path+'run_params.json'
    json_file_dir = Path(json_output_path).parent
    json_file_dir.mkdir(parents=True, exist_ok=True)
    
    # define custom parameters for nanoflowz run
    
    json_params_content = {
        "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
        "rundir": str(Path(run_dir).resolve()),
        "dorado": str(file_paths['dorado_executor']),
        "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
        "polyA": str(polyA_toml_output_path),
        "ref": str(file_paths[organism+'_genome_file']),
        "reference_gtf": str(file_paths[organism+'_'+('basic_' if organism=='human' else '')+'annotation_file']), # for human samples, we'll try with BASIC Gencode annotation
        "gpu_partition": gpu_to_use, # we'll submit jobs to different gpus for increased efficiency
        "gpu_qos": gpu_to_use+'-30min',
        "gpu_time": '30min',
    }
        
    params_file = Path(json_output_path)
        
    with open(params_file, "w") as f:
        json.dump(json_params_content, f, indent=4)

    # run the printed command from any directory under the LOGIN node, under `nextflow` conda environment, created above
    # note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
    cmd = 'nextflow run '+\
    subdirs['nanoflowz_dir']+'main.nf'+\
    ' -params-file '+str(params_file.resolve())+\
    ' -profile conda'+\
    ' -resume'
    print(cmd)